# 03_框架能力

**来源:** [LangChain Docs — Harness capabilities](https://docs.langchain.com/oss/python/deepagents/harness)

Deep Agents 的 Harness 提供了四大类内置能力，用于构建长时间运行、可靠的 Agent：执行环境、上下文管理、委派和引导。此外，Harness Profile 允许将每个模型的配置打包为可复用的捆绑包。


## 1. 执行环境 (Execution Environment)

执行环境是 Agent 行动的场所，包含四个层面：

| 层次 | 说明 |
|------|------|
| **Tools** | 自定义函数、API 和数据库，Agent 可调用 |
| **虚拟文件系统** | 基于可插拔后端的文件操作工具 |
| **文件系统权限** | 声明式访问控制，控制 Agent 可读写路径 |
| **代码执行** | 沙盒 Shell 执行和进程内 JS 解释器 |


In [ ]:
# 创建带有工具的 Agent
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[search, fetch_page, run_query],  # 自定义工具
)


### 1.1 虚拟文件系统 (Virtual Filesystem)

Harness 提供可配置的虚拟文件系统，支持可插拔后端（本地磁盘、内存状态、LangGraph 存储等）。

**内置文件系统操作工具：**

| 工具 | 说明 |
|------|------|
| `ls` | 列出目录中的文件，含大小和修改时间 |
| `read_file` | 读取文件内容，支持 offset/limit，支持多模态文件 |
| `write_file` | 创建新文件 |
| `edit_file` | 执行精确字符串替换（支持全局替换模式） |
| `glob` | 查找匹配模式的文件（如 `**/*.py`） |
| `grep` | 搜索文件内容（文件列表、带上下文的匹配行或计数） |
| `execute` | 运行 Shell 命令（仅沙盒后端可用） |

**支持的多模态文件类型：**

| 类别 | 扩展名 |
|------|--------|
| 图片 | `.png`, `.jpg`, `.jpeg`, `.gif`, `.webp`, `.heic`, `.heif` |
| 视频 | `.mp4`, `.mpeg`, `.mov`, `.avi`, `.flv`, `.mpg`, `.webm`, `.wmv`, `.3gpp` |
| 音频 | `.wav`, `.mp3`, `.aiff`, `.aac`, `.ogg`, `.flac` |
| 文档 | `.pdf`, `.ppt`, `.pptx` |


In [ ]:
# 通过 HarnessProfile 隐藏默认文件系统工具
from deepagents import HarnessProfile, register_harness_profile

register_harness_profile(
    "anthropic:claude-sonnet-4-6",
    HarnessProfile(
        excluded_tools=frozenset({
            "ls", "read_file", "write_file", "edit_file",
            "glob", "grep"
        }),
    ),
)


### 1.2 文件系统权限 (Filesystem Permissions)

支持声明式权限规则，控制 Agent 可读写哪些目录。

- 通过 `permissions=` 参数传入规则列表
- 每条规则指定操作（`"read"`/`"write"`）、路径（glob 模式）和模式（`"allow"`/`"deny"`）
- **首条匹配规则生效**（first-match-wins）
- 若无规则匹配，操作默认允许
- 适用于限制 Agent 在特定目录下操作、保护敏感文件、为子 Agent 分配更窄权限


In [ ]:
# 使用 LocalShellBackend 在本地执行 Shell 命令
from deepagents import create_deep_agent
from deepagents.backends import LocalShellBackend

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[my_tool],
    sandbox=LocalShellBackend(),  # 启用 execute 工具
)


## 2. 上下文管理 (Context Management)

上下文管理控制 Agent 知道什么、能在 Token 限制内运行多长时间、以及在会话间保留什么。

| 层次 | 说明 |
|------|------|
| **Skills** | 按需加载的领域知识，来自技能文件 |
| **Memory** | 持久化指令和偏好，启动时从 `AGENTS.md` 加载 |
| **上下文压缩** | 自动压缩对话历史和大工具结果 |
| **Prompt Caching** | 静态提示段可缓存，加速推理、降低成本 |


### 2.1 技能 (Skills)

Skills 提供专业化工作流和领域知识。

- 遵循 **Agent Skills 标准**
- 每个 Skill 是一个目录，包含 `SKILL.md` 文件（含指令和元数据）
- 可包含额外脚本、参考文档、模板等资源
- **渐进式加载** — 仅在 Agent 确定有用时才读取
- 减少 Token 消耗，模块化 Agent 能力


### 2.2 记忆 (Memory)

持久化记忆文件为 Agent 提供跨会话的额外上下文。

- 使用 `AGENTS.md` 文件提供持久化上下文
- 始终加载（不像 Skills 那样渐进式加载）
- 通过 `memory` 参数传入文件路径列表
- 存储在 Agent 后端（StateBackend、StoreBackend 或 FilesystemBackend）
- Agent 可根据交互、反馈和识别到的模式更新记忆


In [ ]:
# 为 Agent 配置记忆文件
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[my_tool],
    memory=["project/AGENTS.md"],  # 持久化记忆
)


### 2.3 上下文压缩与卸载 (Summarization & Context Offloading)

Harness 管理上下文以让 Agent 处理长时间运行的任务：

- **输入上下文** — 系统提示、记忆、技能和工具提示塑造 Agent 启动时知道的内容
- **压缩** — 内置卸载和摘要功能将上下文保持在窗口限制内
- **隔离** — 子 Agent 隔离繁重工作，仅返回结果
- **长期记忆** — 通过虚拟文件系统跨线程持久化存储


### 2.4 提示缓存 (Prompt Caching)

- 对 Anthropic 模型，`create_deep_agent` 自动对系统提示的静态部分应用提示缓存
- 覆盖基础 Agent 指令、记忆和技能内容
- 默认启用，无需额外配置
- 减少长时间运行 Agent 的延迟和成本


## 3. 委派 (Delegation)

委派组件使 Agent 能将大问题分解为更小、可并行化的任务单元。

| 层次 | 说明 |
|------|------|
| **任务规划** | 内置 `write_todos` 工具，用于结构化任务跟踪 |
| **子 Agent** | 临时子 Agent，处理隔离的子任务 |


### 3.1 任务规划 (Task Planning)

`write_todos` 工具允许 Agent 维护结构化的任务列表：

- 跟踪多个任务，状态为 `"pending"`、`"in_progress"`、`"completed"`
- 持久化存储在 Agent 状态中
- 帮助 Agent 组织复杂多步工作


### 3.2 子 Agent (Subagents)

主 Agent 可以创建临时子 Agent 来处理隔离的多步任务。

**子 Agent 的优势：**

| 特性 | 说明 |
|------|------|
| **上下文隔离** | 子 Agent 的工作不会污染主 Agent 的上下文 |
| **并行执行** | 多个子 Agent 可同时运行 |
| **专业化** | 子 Agent 可有不同的工具和配置 |
| **Token 效率** | 大型子任务上下文被压缩为单一结果 |

**工作方式：**
1. 主 Agent 有 `task` 工具
2. 调用时创建新的 Agent 实例，自带上下文
3. 子 Agent 自主执行直到完成
4. 向主 Agent 返回单一最终报告
5. 子 Agent 是无状态的（无法发送多条消息回来）


## 4. 引导 (Steering)

引导组件赋予人类在运行时控制 Agent 行为的能力。


### 4.1 人在回路 (Human-in-the-Loop)

Harness 可以在指定的工具调用处暂停 Agent 执行，等待人工批准或修改。

- 通过 `interrupt_on` 参数配置
- 格式：`interrupt_on={"edit_file": True}` —— 每次编辑前暂停
- 可在提示时提供审批消息或修改工具输入
- **应用场景：** 破坏性操作的安全门、昂贵 API 调用前的验证、交互式调试


In [ ]:
# 配置人在回路：编辑文件前暂停审批
from deepagents import create_deep_agent

agent = create_deep_agent(
    model="anthropic:claude-sonnet-4-6",
    tools=[my_tool],
    interrupt_on={"edit_file": True},  # 编辑前暂停
)


## 5. Harness Profile

Harness Profile 是声明式配置包，每当选择某供应商或模型时自动应用。

- 在供应商名称（`"openai"`）或 `provider:model` 键（`"openai:gpt-5.4"`）下注册
- `create_deep_agent` 在解析模型时查找并应用 Profile
- 供应商级和模型级 Profile 在解析时合并
- **用途：** 包装每供应商/每模型的默认值（系统提示调整、工具覆盖、中间件）


In [ ]:
# 注册和使用 HarnessProfile
from deepagents import HarnessProfile, register_harness_profile

# 为特定模型注册配置
register_harness_profile(
    "openai:gpt-5.4",
    HarnessProfile(
        excluded_tools=frozenset({"execute"}),
    ),
)

# 创建 Agent 时自动应用匹配的 Profile
agent = create_deep_agent(
    model="openai:gpt-5.4",
    tools=[my_tool],
)


---
**参考链接**

- [Harness capabilities — LangChain Docs](https://docs.langchain.com/oss/python/deepagents/harness)
- [Documentation Index](https://docs.langchain.com/llms.txt)
